# Transformer Decoder

## Code

In [1]:
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn


## Testing

In [2]:
# input parameters
N = 3
M = 4
batch_size = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float32

# Decoder Layer parameters
d_model = 4
nhead = 2
dim_feedforward = 64
dropout = 0.2
layer_norm_eps = 1e-5
batch_first = True
norm_first = True
bias = True

src_mask = None  # torch.rand((N, N), device=device, dtype=torch.bool)
tgt_mask = None  # torch.rand((M, M), device=device, dtype=torch.bool)
memory_mask = None  # torch.rand((M, N), device=device, dtype=torch.bool)


## Transformer decoder parameters
num_enc_layers = 3
num_dec_layers = 5
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [3]:
torch.manual_seed(0)
x = torch.randn(batch_size, N, d_model, device=device, dtype=dtype)
tgt = torch.randn(batch_size, M, d_model, device=device, dtype=dtype)

In [4]:
init_seed = 42  # avoide weights initialization randomness effects
train_seed = 24  # avoid dropout randomness effects

In [5]:
torch.manual_seed(init_seed)
tf = mynn.Transformer(
    d_model,
    nhead,
    num_encoder_layers=num_enc_layers,
    num_decoder_layers=num_dec_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation_cls=nn.GELU,
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)

torch.manual_seed(init_seed)
nn_tf = nn.Transformer(
    d_model,
    nhead,
    num_encoder_layers=num_enc_layers,
    num_decoder_layers=num_dec_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation="gelu",
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
    batch_first=batch_first,
)

tf.load_weights_from_torch_transformer(nn_tf)

/Users/abeldiaz/Documents/learn/cs/3-Machine Learning/ml-notebook/.venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


### Evaluation

In [6]:
tf.eval()
tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-0.3377,  1.1564,  0.6426, -1.4614],
         [ 0.5067, -0.1441,  1.1709, -1.5334],
         [ 0.7388, -0.5009,  1.1410, -1.3789],
         [-0.8633,  1.4429,  0.4191, -0.9986]],

        [[-1.4732,  0.7872, -0.3551,  1.0411],
         [ 0.0182,  0.1673, -1.4989,  1.3134],
         [ 0.4729, -1.7173,  0.7847,  0.4597],
         [ 1.0257, -1.4175, -0.4606,  0.8524]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)

In [7]:
nn_tf.eval()
nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-0.3377,  1.1564,  0.6426, -1.4614],
         [ 0.5067, -0.1441,  1.1709, -1.5334],
         [ 0.7388, -0.5009,  1.1410, -1.3789],
         [-0.8633,  1.4429,  0.4191, -0.9986]],

        [[-1.4732,  0.7872, -0.3551,  1.0411],
         [ 0.0182,  0.1673, -1.4989,  1.3134],
         [ 0.4729, -1.7173,  0.7847,  0.4597],
         [ 1.0257, -1.4175, -0.4606,  0.8524]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)

### Training

In [8]:
mse = torch.nn.MSELoss()

In [9]:
torch.manual_seed(train_seed)
nn_tf.train()
out = nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)
print(out)
loss = mse(out, tgt)
loss.backward()
optimizer = torch.optim.SGD(nn_tf.parameters(), lr=1e-3)
optimizer.step()
nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-0.3625,  1.3143,  0.4429, -1.3946],
         [-0.2408,  1.6190, -0.2583, -1.1199],
         [-0.7150,  1.3473,  0.5422, -1.1745],
         [-0.8840,  1.6148,  0.0492, -0.7800]],

        [[-0.9519,  0.6397,  1.3041, -0.9919],
         [-0.1098,  1.2935, -1.4903,  0.3065],
         [ 0.7980, -0.8126,  1.1698, -1.1552],
         [ 1.5262, -1.2809, -0.1230, -0.1222]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)


tensor([[[ 0.3857,  1.2009, -0.0380, -1.5493],
         [ 0.9259, -1.6431,  0.6610,  0.0571],
         [-0.3477, -1.4231,  1.2620,  0.5094],
         [-1.0903,  1.5900, -0.5262,  0.0249]],

        [[-1.2652,  1.4876, -0.3918,  0.1678],
         [ 1.2061, -0.0288, -1.5508,  0.3731],
         [ 0.4485, -1.5926,  1.1222,  0.0228],
         [ 1.1435, -0.6757, -1.2650,  0.7971]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)

In [ ]:
torch.manual_seed(train_seed)
tf.train()
out = tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)
print(out)
loss = mse(out, tgt)
loss.backward()
optimizer = torch.optim.SGD(tf.parameters(), lr=1e-3)
optimizer.step()
tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-0.3625,  1.3143,  0.4429, -1.3946],
         [-0.2408,  1.6190, -0.2583, -1.1199],
         [-0.7150,  1.3473,  0.5422, -1.1745],
         [-0.8840,  1.6148,  0.0492, -0.7800]],

        [[-0.9519,  0.6397,  1.3041, -0.9919],
         [-0.1098,  1.2935, -1.4903,  0.3065],
         [ 0.7980, -0.8126,  1.1698, -1.1552],
         [ 1.5262, -1.2809, -0.1230, -0.1222]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)


tensor([[[ 0.3867,  1.2010, -0.0394, -1.5490],
         [ 0.9259, -1.6431,  0.6607,  0.0574],
         [-0.3461, -1.4238,  1.2621,  0.5083],
         [-1.0899,  1.5902, -0.5266,  0.0247]],

        [[-1.2651,  1.4875, -0.3921,  0.1681],
         [ 1.2061, -0.0292, -1.5507,  0.3734],
         [ 0.4490, -1.5928,  1.1217,  0.0228],
         [ 1.1426, -0.6769, -1.2643,  0.7984]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)